### Tools


Models can request to call tools that perform tasks such as fetching data from a database , searching the web or runnung code. Tools are pairings of 

1. A schema , including the name of the tool , a description, and/or argument definitions (often a json schema)
2. A function or coroutine to execute


In [5]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response= model.invoke("why do parrots talk")
response

AIMessage(content='<think>\nOkay, so why do parrots talk? Let me start by recalling what I know about parrots. I know they\'re known for mimicking human speech, but why exactly do they do that? Maybe it\'s for communication. But with whom? Other parrots, or humans?\n\nI remember reading that parrots are highly intelligent birds. Their ability to mimic sounds could be related to their social nature. Maybe they learn to talk by listening to humans around them. But why would they evolve this ability? In the wild, do parrots mimic other sounds besides human speech?\n\nAlso, there\'s the aspect of vocal learning. Some animals can imitate sounds, like some whales or certain birds. Parrots must have specific brain structures that allow them to process and replicate sounds. I think the term is "vocal learning," which is a trait in only a few species.\n\nAnother angle is the purpose of talking. Do they use it to interact with humans, to get attention, or to bond? Maybe it\'s a form of social bo

In [18]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location""" # it is a doc string which is very important to write 
    return f"the weather of{location} is sunny"

model_with_tool = model.bind_tools([get_weather]) #  binding the tool with llm 


In [19]:
#calling the model
response = model_with_tool.invoke("what's the weather in chennai ?")
print(response)
for tool_call in response.tool_calls:
#view tool calls made by the model
    print(f"Tool :{tool_call['name']}")
    print(f"Args: {tool_call['args']}")
#view tool calls made by the model

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call this function with "Chennai" as the location. I\'ll make sure the arguments are correctly formatted in JSON within the tool_call tags.\n', 'tool_calls': [{'id': 'jc0th8874', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 154, 'total_tokens': 249, 'completion_time': 0.147234564, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.015358411, 'prompt_tokens_details': None, 'queue_time': 0.184023985, 'total_time': 0.162592975}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider

Another way 

In [ ]:

from langchain.agents import create_agent

def get_weather_city(city:str)->str:
    """get the weather for a city"""
    return f"the weather in {city} is sunny"

agent= create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools =[get_weather_city],
    system_prompt="you are helpful assistant",
)
agent

response=agent.invoke({"messages":[{"role": "user","content":"what is the weather like in new york ?"}]})
response

### Tool Execution Loop

In [20]:
# Step 1: Model generates tool calls
messages= [{"role":"user" , "content":"what's the weather in boston"}]
ai_msg= model_with_tool.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tool and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3 : Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

#current weather in boston is 72*F and sunny

The weather in Boston is sunny.


In [21]:
messages

[{'role': 'user', 'content': "what's the weather in boston"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location they mentioned. I\'ll call the function with "Boston" as the argument. Let me make sure the JSON is formatted correctly. The name should be get_weather and the arguments should have location as Boston. Yep, that looks right.\n', 'tool_calls': [{'id': 'k3pz9bjzp', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 153, 'total_tokens': 256, 'completion_time': 0.18206954, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.006073082, 'prompt_tokens_details': None, 'queue_time': 0.652297687, 'total_time': 0.188142622}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpri